# Classification Validation

This notebook validates the two classified datasets produced in the classification notebook. The goal is to inspect the extracted groups, evaluate overlap between them, and remove ambiguous videos that were assigned to both categories.

The result of this notebook is a clean comparison dataset for the final audience-conversation analysis.


## Load The MongoDB-Exported Classification Datasets

These files are created in `classification.ipynb` and stored in the `data` folder.


In [21]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

employee = pd.read_json("../data/employee-based-content.json", lines=True)
product = pd.read_json("../data/product-based-content.json", lines=True)


## Dataset Overview

In [22]:
print("Employer-branding data size:", len(employee))
print("Product-technology data size:", len(product))


Employer-branding data size: 3120
Product-technology data size: 4605


In [23]:
employee.head()


,_id,channelId,company,title,description,tags,publishedAt,views,likes,comments
0,jiHmzKeqeT8,UCLAKVzA5WEadvys9CKOksQg,Adecco SA,Klåss Clerkx - CEO for One Month 2016 at Adecc...,The #CEO1Month selection process started with ...,"[CEO1MONTH, Adecco Way to Work, bootcamp]",2016-05-25T15:51:58.000Z,156,1.0000,0.0000
1,1p9NjGCEieg,UCrSSdGg-aH_HgCHtsnoonIg,Adecco SA,Adecco Thailand WayToWork 2014,กะเทาะเปลือกเด็กจบใหม่ ครั้งที่ 2 กิจกรรมเพื่อ...,"[Way2WorkTh, AdeccoThailand, WayToWork, HRtwt,...",2014-04-30T11:10:20.000Z,1249,5.0000,1.0000
2,bmvCxO3zvsQ,UCrSSdGg-aH_HgCHtsnoonIg,Adecco SA,Adecco Thailand Career Up - Marketing Jobs,How to Introduce Yourself in English for Marke...,"[Marketing, Adecco, Jobs Seeker, Jobs]",2011-09-23T03:24:05.000Z,4535,6.0000,3.0000
3,61taXGTki5g,UCLAKVzA5WEadvys9CKOksQg,Adecco SA,Jobs bij TNT: Customer Service medewerkers in ...,"Je bent klantvriendelijk, goed in communicatie...","[Jobs, TNT, Customer Service, Machelen, België]",2016-05-17T11:15:11.000Z,131,1.0000,0.0000
4,W33uEUbULRs,UC7wvKWegeU9FGzEcVhJVoDw,ABB Ltd,Anna – Success begins with you,"Success begins with you because, at ABB, you c...","[ABB Ltd, ABB Careers, Technology, Engineering]",2016-07-18T14:27:22.000Z,379,5.0000,1.0000


In [24]:
product.head()


,_id,channelId,company,title,description,tags,publishedAt,views,likes,comments
0,EvjO6DIurLk,UCXyz4vATE49KlC_7vUY-bdw,Cembra Money Bank AG,Cembra Money Bank Famiglia con divano Italiano,Rebranding: Campagna Autunno 2013,"[Cembra Money Bank, Familie, Sofa, GE Money Ba...",2013-11-11T10:26:04.000Z,27387,0.0000,0.0000
1,jMlOv0Z5GpE,UCJ8IXb4xgt85gSKMpxXfpVQ,Bossard Holding AG,So funktioniert die Wärme-Einbettung eines Tap...,In warmem Kunststoff feste Verbindungen schaff...,"[kvt, fastening, kvt-fastening, Verbindungsele...",2015-06-22T07:47:56.000Z,293,NaN,NaN
2,61taXGTki5g,UCLAKVzA5WEadvys9CKOksQg,Adecco SA,Jobs bij TNT: Customer Service medewerkers in ...,"Je bent klantvriendelijk, goed in communicatie...","[Jobs, TNT, Customer Service, Machelen, België]",2016-05-17T11:15:11.000Z,131,1.0000,0.0000
3,W33uEUbULRs,UC7wvKWegeU9FGzEcVhJVoDw,ABB Ltd,Anna – Success begins with you,"Success begins with you because, at ABB, you c...","[ABB Ltd, ABB Careers, Technology, Engineering]",2016-07-18T14:27:22.000Z,379,5.0000,1.0000
4,jOWd1QNgr7c,UCf6fYOaOmIGuLPSGncxELNA,Baloise Holding AG,Baloise Jobcast #4: Personal Branding,Wie präsentiere ich meine Eigenmarke? In diese...,"[bewerben, auftritt, image, baloise, podcast, ...",2016-05-02T14:47:55.000Z,47,1.0000,0.0000


In [25]:
overview = pd.DataFrame(
    {
        "n_videos": [len(employee), len(product)],
        "unique_companies": [employee["company"].nunique(), product["company"].nunique()],
        "missing_values": [employee.isna().sum().sum(), product.isna().sum().sum()],
    },
    index=["employer_branding", "product_technology"],
)

overview


,n_videos,unique_companies,missing_values
employer_branding,3120,94,677
product_technology,4605,110,698


In [26]:
employee["company"].value_counts().head(10)


company
Adecco SA                 509
Nestle SA                 332
Swatch Group AG/The       155
Novartis AG               139
Roche Holding AG          137
Helvetia Holding AG       129
UBS Group AG              119
Credit Suisse Group AG    113
ABB Ltd                   112
Vontobel Holding AG       112
Name: count, dtype: int64

In [27]:
product["company"].value_counts().head(10)


company
Nestle SA                         676
ABB Ltd                           413
Helvetia Holding AG               233
Roche Holding AG                  222
UBS Group AG                      206
Georg Fischer AG                  204
Logitech International SA         135
Novartis AG                       131
Credit Suisse Group AG            130
Sunrise Communications Group A    124
Name: count, dtype: int64

The datasets cover a large number of firms, but some channels contribute far more videos than others. This concentration should be considered when interpreting later results.


## Time Range

In [28]:
employee["publishedAt"] = pd.to_datetime(employee["publishedAt"], format="mixed", errors="coerce")
product["publishedAt"] = pd.to_datetime(product["publishedAt"], format="mixed", errors="coerce")


In [29]:
print(
    "Employer-branding publication range:",
    employee["publishedAt"].min().strftime("%Y-%m-%d"),
    "to",
    employee["publishedAt"].max().strftime("%Y-%m-%d"),
)
employee["publishedAt"].dt.year.value_counts().sort_index()


Employer-branding publication range: 2007-12-10 to 2024-03-28


publishedAt
2007      1
2008      1
2009     16
2010    131
2011    256
2012    407
2013    566
2014    641
2015    641
2016    427
2018      1
2019      1
2020      1
2021      1
2022     12
2023     13
2024      4
Name: count, dtype: int64

In [30]:
print(
    "Product-technology publication range:",
    product["publishedAt"].min().strftime("%Y-%m-%d"),
    "to",
    product["publishedAt"].max().strftime("%Y-%m-%d"),
)
product["publishedAt"].dt.year.value_counts().sort_index()


Product-technology publication range: 2007-11-19 to 2022-12-01


publishedAt
2007       1
2008       5
2009      75
2010     174
2011     413
2012     470
2013     800
2014     951
2015    1008
2016     701
2022       7
Name: count, dtype: int64

In [31]:
missing_summary = pd.DataFrame(
    {
        "employer_branding": employee.isna().sum(),
        "product_technology": product.isna().sum(),
    }
)

missing_summary


,employer_branding,product_technology
_id,0,0
channelId,33,7
company,33,7
title,0,0
description,0,0
tags,0,0
publishedAt,0,0
views,0,0
likes,156,200
comments,455,484


## Overlap Analysis

Videos that appear in both datasets are ambiguous for the later comparison because they carry both employer-branding and product-technology signals.


In [32]:
employee_ids = set(employee["_id"])
product_ids = set(product["_id"])
overlap_ids = employee_ids.intersection(product_ids)

print("Number of overlapping videos:", len(overlap_ids))


Number of overlapping videos: 956


In [33]:
overlap_df = employee[employee["_id"].isin(overlap_ids)].copy()
overlap_sample = overlap_df.sample(min(30, len(overlap_df)), random_state=42)

overlap_sample[["title", "company", "tags"]]


,title,company,tags
1347,Warehouse + NDIC Supply Chain | What You Do Ma...,Nestle SA,"[Nestle, Nestle USA, Nestle on Youtube, Nestle..."
2912,Straumann® CARES® - Wax Abutment,Straumann Holding AG,"[Wax abutment, Cares Visual, abutment, abutmen..."
1819,Iba Na – Carbonara | NESTLÉ All Purpose Cream ...,Nestle SA,"[nestle, nestle philippines, nestle ph, nestle..."
1866,Arun Sundararajan on how work will be transformed,Swiss Re AG,"[professor, New York University, Leonard N. St..."
2263,Simone talks about life at SGS,SGS SA,"[testimonial, employer branding, video, sgs pe..."
640,Découvrez le nouveau site du groupe Adecco France,Adecco SA,"[Adecco, groupe Adecco France, emploi, recrute..."
938,PHASETREAT® Demulsifiers - An ecofriendly solu...,Clariant AG,"[demulsifiers, oil, innovation, Clariant (Busi..."
787,"Moving from ""Just in Case"" to ""Just in Time"" i...",Ascom Holding AG,"[Ascom, Ascom Network Testing, TEMS, TEMS Solu..."
379,New Launcher with TEMS Investigation – Fast ac...,Ascom Holding AG,"[Ascom, Ascom Network Testing, TEMS, TEMS Inve..."
1133,ABB motion control products - Product Synchron...,ABB Ltd,"[ABB, motion control, motion control products,..."


A sample of the overlapping videos should be inspected manually. These cases typically reflect videos that mention both jobs or employees and products or technology, which makes them ambiguous for a clean two-group comparison.


## Remove Overlapping Videos

The final audience-conversation notebook should compare only uniquely classified videos. The overlap is therefore removed from both datasets before saving the clean files.


In [34]:
employee_clean = employee[~employee["_id"].isin(overlap_ids)].copy()
product_clean = product[~product["_id"].isin(overlap_ids)].copy()

summary = pd.DataFrame(
    {
        "original": [len(employee), len(product)],
        "clean": [len(employee_clean), len(product_clean)],
    },
    index=["employer_branding", "product_technology"],
)

summary["removed_as_overlap"] = summary["original"] - summary["clean"]
summary


,original,clean,removed_as_overlap
employer_branding,3120,2164,956
product_technology,4605,3649,956


In [35]:
employee_clean.to_json("../data/employee-based-content-clean.json", orient="records", lines=True)
product_clean.to_json("../data/product-based-content-clean.json", orient="records", lines=True)


After removing overlap, the project has two clean datasets ready for the final research-question analysis:

- `data/employee-based-content-clean.json`
- `data/product-based-content-clean.json`
